![Supabase](https://raw.githubusercontent.com/supabase/supabase/master/packages/common/assets/images/supabase-logo-wordmark--light.svg)


#Amazon S3 to Supabase Storage migration guide

Run the cells top to bottom. Only **Step 1** needs editing.

1. Fill in the template &nbsp;2. Count files (nothing moves) &nbsp;3. Create bucket &nbsp;4. Migrate &nbsp;5. Verify

Get your Supabase keys at **Storage &rarr; S3 Configuration &rarr; Access keys**.

In [ ]:
#@title Select the transfer `Method` & install required resources. { display-mode: "form" }
method = 'boto3 (Python)' #@param ["boto3 (Python)", "rclone"]

if method == 'boto3 (Python)':
  !pip install -q --upgrade boto3 botocore
  import boto3
  print("Installed boto3 to migrate Amazon S3 to Supabase Storage")
else:
  !curl -s https://rclone.org/install.sh | sudo bash &>log
  print("rclone installed to migrate Amazon S3 to Supabase Storage")

## Step 1: Set the environment Variables

Replace every `<...>` value below.

In [ ]:
#Source (Amazon S3):
%env AWS_ACCESS_KEY_ID=<YOUR_AWS_ACCESS_KEY_ID>
%env AWS_SECRET_ACCESS_KEY=<YOUR_AWS_SECRET_ACCESS_KEY>
%env AWS_REGION=us-east-1
%env SOURCE_BUCKET=<YOUR_S3_BUCKET_NAME>
#Only for temporary ASIA... keys. Leave commented out for normal AKIA... keys.
#%env AWS_SESSION_TOKEN=<YOUR_AWS_SESSION_TOKEN>

#Destination (Supabase Storage):
%env SUPABASE_S3_ENDPOINT=https://<YOUR_PROJECT_REF>.storage.supabase.co/storage/v1/s3
%env SUPABASE_REGION=<YOUR_PROJECT_REGION>
%env SUPABASE_ACCESS_KEY_ID=<YOUR_SUPABASE_ACCESS_KEY_ID>
%env SUPABASE_SECRET_ACCESS_KEY=<YOUR_SUPABASE_SECRET_ACCESS_KEY>
%env TARGET_BUCKET=<YOUR_SUPABASE_BUCKET_NAME>

#Optional. Leave empty to migrate the whole bucket / keep keys unchanged.
%env SOURCE_PREFIX=
%env TARGET_PREFIX=

## Step 2: Connect

Checks the credentials and warns about any `<...>` you forgot to replace.

A `403` on the source means the key is valid but not allowed to read that bucket. Attach this to
the IAM user, replacing the bucket name:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::<YOUR_S3_BUCKET_NAME>"
    },
    {
      "Effect": "Allow",
      "Action": "s3:GetObject",
      "Resource": "arn:aws:s3:::<YOUR_S3_BUCKET_NAME>/*"
    }
  ]
}
```

In [ ]:
#@title #Step 2: Connect to both buckets { display-mode: "form" }
import os, boto3
from botocore.config import Config
from botocore.exceptions import ClientError, EndpointConnectionError

SOURCE_BUCKET = os.environ["SOURCE_BUCKET"]
SOURCE_PREFIX = os.environ.get("SOURCE_PREFIX", "").lstrip("/")
TARGET_BUCKET = os.environ["TARGET_BUCKET"]
TARGET_PREFIX = os.environ.get("TARGET_PREFIX", "").strip().strip("/")

REQUIRED = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION", "SOURCE_BUCKET",
            "SUPABASE_S3_ENDPOINT", "SUPABASE_REGION", "SUPABASE_ACCESS_KEY_ID",
            "SUPABASE_SECRET_ACCESS_KEY", "TARGET_BUCKET"]
todo = [k for k in REQUIRED if "<" in os.environ.get(k, "")]
if todo:
  raise SystemExit("Still a template -- replace these in Step 1: " + ", ".join(todo))

# Malformed credentials fail as SignatureDoesNotMatch, which reads like a
# permissions problem. Catch the common paste mistakes up front instead.
AKID = os.environ["AWS_ACCESS_KEY_ID"].strip()
ASEC = os.environ["AWS_SECRET_ACCESS_KEY"].strip()
TOKEN = os.environ.get("AWS_SESSION_TOKEN", "").strip()

for name, val, size in [("AWS_ACCESS_KEY_ID", AKID, 20), ("AWS_SECRET_ACCESS_KEY", ASEC, 40)]:
  if val != os.environ[name]:
    print(f"[warn] {name} had surrounding whitespace -- trimmed")
  if val[:1] in "\"'" or val[-1:] in "\"'":
    print(f"[warn] {name} is wrapped in quotes -- remove them, %env needs a bare value")
  if len(val) != size:
    print(f"[warn] {name} is {len(val)} chars, expected {size} -- truncated or mistyped")
if AKID.startswith("ASIA") and not TOKEN:
  print("[warn] ASIA... is a temporary key -- also set AWS_SESSION_TOKEN in Step 1")


def _supabase_config():
  # path addressing: Supabase has no virtual-hosted buckets.
  # when_required: botocore >=1.36 adds a CRC32 header Supabase rejects.
  base = dict(signature_version="s3v4", s3={"addressing_style": "path"},
              retries={"max_attempts": 10, "mode": "adaptive"}, max_pool_connections=32)
  try:
    return Config(request_checksum_calculation="when_required",
                  response_checksum_validation="when_required", **base)
  except TypeError:
    return Config(**base)


src = boto3.client("s3",
                   aws_access_key_id=AKID,
                   aws_secret_access_key=ASEC,
                   aws_session_token=TOKEN or None,
                   region_name=os.environ["AWS_REGION"].strip(),
                   config=Config(signature_version="s3v4",
                                 retries={"max_attempts": 10, "mode": "adaptive"},
                                 max_pool_connections=32))

dst = boto3.client("s3",
                   endpoint_url=os.environ["SUPABASE_S3_ENDPOINT"],
                   aws_access_key_id=os.environ["SUPABASE_ACCESS_KEY_ID"],
                   aws_secret_access_key=os.environ["SUPABASE_SECRET_ACCESS_KEY"],
                   region_name=os.environ["SUPABASE_REGION"],
                   config=_supabase_config())


def human(n):
  for unit in ("B", "KB", "MB", "GB", "TB", "PB"):
    if abs(n) < 1024 or unit == "PB":
      return f"{n:,.1f} {unit}" if unit != "B" else f"{n:,.0f} B"
    n /= 1024.0


def dest_key(key):
  k = key.lstrip("/")
  if SOURCE_PREFIX and k.startswith(SOURCE_PREFIX):
    k = k[len(SOURCE_PREFIX):].lstrip("/")
  return f"{TARGET_PREFIX}/{k}" if TARGET_PREFIX else k


ok = True
try:
  # Probe the operations the migration actually uses. HeadBucket is a poor test:
  # it ignores s3:prefix conditions, so a prefix-scoped key fails it yet migrates fine.
  probe = src.list_objects_v2(Bucket=SOURCE_BUCKET, Prefix=SOURCE_PREFIX, MaxKeys=1)
  print(f"[ok]   source: can list s3://{SOURCE_BUCKET}/{SOURCE_PREFIX}")
  first = [o["Key"] for o in probe.get("Contents", [])]
  if first:
    src.head_object(Bucket=SOURCE_BUCKET, Key=first[0])
    print("[ok]   source: can read objects")
  else:
    print("[warn] source: no objects under that prefix")
except (ClientError, EndpointConnectionError) as e:
  ok = False
  print(f"[FAIL] source: {e}")
  code = getattr(e, "response", {}).get("Error", {}).get("Code", "")
  region = (getattr(e, "response", {}).get("ResponseMetadata", {})
            .get("HTTPHeaders", {}).get("x-amz-bucket-region"))

  if code == "SignatureDoesNotMatch":
    print("       AWS_SECRET_ACCESS_KEY does not match AWS_ACCESS_KEY_ID.")
    print("       This is the secret, not a permissions problem -- the IAM policy")
    print("       will not help. Re-copy the secret, or make a new key pair at")
    print("       IAM > Users > your user > Security credentials > Create access key.")
    print(f"       Currently using key id {AKID[:4]}...{AKID[-4:]} ({len(AKID)} chars),")
    print(f"       secret of {len(ASEC)} chars (expected 40).")
  elif code in ("InvalidAccessKeyId", "InvalidClientTokenId"):
    print("       AWS_ACCESS_KEY_ID does not exist -- deleted, deactivated, or a typo.")
  elif code in ("ExpiredToken", "ExpiredTokenException", "InvalidToken"):
    print("       Temporary credentials expired -- refresh them and set AWS_SESSION_TOKEN.")
  elif code in ("NoSuchBucket",):
    print(f"       Bucket '{SOURCE_BUCKET}' does not exist in this account.")
  elif region and region != os.environ["AWS_REGION"].strip():
    print(f"       Bucket is in {region}, not {os.environ['AWS_REGION']} -- fix AWS_REGION.")
  elif code in ("AccessDenied", "403", "AllAccessDisabled"):
    print("       Key is valid but lacks s3:ListBucket / s3:GetObject on this bucket,")
    print("       or SOURCE_BUCKET is misspelled / in another AWS account.")
    try:
      who = boto3.client("sts", aws_access_key_id=AKID, aws_secret_access_key=ASEC,
                         aws_session_token=TOKEN or None,
                         region_name=os.environ["AWS_REGION"].strip()).get_caller_identity()
      print(f"       Authenticated as {who['Arn']} (account {who['Account']})")
    except Exception:
      pass
    print("       Attach the IAM policy shown above this cell.")
  else:
    print(f"       Unexpected error code '{code}' -- see the troubleshooting table.")

try:
  dst.list_buckets()
  print("[ok]   destination")
except (ClientError, EndpointConnectionError) as e:
  ok = False
  print(f"[FAIL] destination: {e}")

print("\nReady." if ok else "\nSee the troubleshooting table at the bottom.")

## Step 3: Count the files (dry run)

Lists metadata only. **Nothing is downloaded, moved or deleted.** Writes `s3_inventory.csv`,
which Step 5 reads.

In [ ]:
#@title #Step 3: Count what will be migrated (no data is transferred) { display-mode: "form" }
MAX_FILE_MB = 50 #@param {type:"number"}
ASSUMED_MBPS = 40 #@param {type:"number"}

import csv, time
from collections import Counter, defaultdict

MANIFEST = "s3_inventory.csv"
COLD = {"GLACIER", "DEEP_ARCHIVE", "GLACIER_IR"}
max_bytes = int(MAX_FILE_MB * 1024 * 1024)

rows, total_bytes, placeholders = [], 0, 0
by_class, hist = Counter(), defaultdict(int)
oversize, cold = [], []

print(f"Listing s3://{SOURCE_BUCKET}/{SOURCE_PREFIX} ...")
t0 = time.time()
for page in src.get_paginator("list_objects_v2").paginate(Bucket=SOURCE_BUCKET,
                                                          Prefix=SOURCE_PREFIX):
  for obj in page.get("Contents", []):
    key, size = obj["Key"], obj["Size"]
    cls = obj.get("StorageClass", "STANDARD")

    if key.endswith("/") and size == 0:      # console-created folder marker
      placeholders += 1
      continue

    total_bytes += size
    by_class[cls] += 1
    for label, limit in [("0 B", 1), ("< 1 MB", 1024 ** 2), ("1-6 MB", 6 * 1024 ** 2),
                         ("6-50 MB", 50 * 1024 ** 2), ("50 MB-1 GB", 1024 ** 3)]:
      if size < limit:
        hist[label] += 1
        break
    else:
      hist["> 1 GB"] += 1

    if size > max_bytes:
      oversize.append((key, size))
    if cls in COLD:
      cold.append((key, cls))

    rows.append({"key": key, "dest_key": dest_key(key), "size": size,
                 "storage_class": cls, "last_modified": obj["LastModified"].isoformat()})

n = len(rows)
secs = int(total_bytes / (ASSUMED_MBPS * 1024 * 1024)) if n else 0
h, rem = divmod(secs, 3600)
eta = f"{h}h {rem // 60}m" if h else (f"{rem // 60}m" if rem >= 60 else f"{rem}s")

print("\n" + "=" * 56)
print("INVENTORY  (dry run -- nothing transferred)")
print("=" * 56)
print(f"Files to move   {n:,}")
print(f"Total size      {human(total_bytes)}")
print(f"Folder markers  {placeholders:,} skipped")
print(f"Est. transfer   ~{eta} at {ASSUMED_MBPS} MB/s")

print("\nSize distribution")
for label in ["0 B", "< 1 MB", "1-6 MB", "6-50 MB", "50 MB-1 GB", "> 1 GB"]:
  if hist[label]:
    print(f"  {label:<14} {hist[label]:>9,}")

print("\nStorage class")
for cls, cnt in by_class.most_common():
  print(f"  {cls:<14} {cnt:>9,}")

print("\nBlockers")
if oversize:
  print(f"  {len(oversize):,} file(s) over {MAX_FILE_MB} MB will FAIL -- raise the bucket limit:")
  for key, size in sorted(oversize, key=lambda x: -x[1])[:5]:
    print(f"    {human(size):>11}  {key[:58]}")
else:
  print(f"  None over {MAX_FILE_MB} MB.")
if cold:
  print(f"  {len(cold):,} file(s) in Glacier -- restore in S3 first:")
  for key, cls in cold[:5]:
    print(f"    {cls:<13}  {key[:58]}")
else:
  print("  None archived.")

if rows:
  with open(MANIFEST, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
  print(f"\nWrote {MANIFEST} ({n:,} rows).")

## Step 4: Create the bucket

Makes a **private** bucket. For public files or a custom size limit, use
**Storage &rarr; Buckets** in the dashboard instead.

In [ ]:
#@title #Step 4: Create the Supabase bucket if it does not exist { display-mode: "form" }
existing = [b["Name"] for b in dst.list_buckets().get("Buckets", [])]
print("Buckets: " + (", ".join(existing) if existing else "(none)"))

if TARGET_BUCKET in existing:
  print(f"\n'{TARGET_BUCKET}' already exists.")
else:
  dst.create_bucket(Bucket=TARGET_BUCKET)
  print(f"\nCreated private bucket '{TARGET_BUCKET}'.")

## Step 5: Run the migration

Resumable: progress is saved to `migration_state.jsonl`, so rerunning the cell picks up where it
stopped and retries only what failed. Tick `DRY_RUN` to rehearse without writing.

In [ ]:
#@title #Step 5: Running the Migration: { display-mode: "form" }
DRY_RUN = False #@param {type:"boolean"}
MAX_WORKERS = 8 #@param {type:"integer"}
MULTIPART_THRESHOLD_MB = 8 #@param {type:"number"}
RETRIES = 3 #@param {type:"integer"}

import csv, json, mimetypes, os, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from boto3.s3.transfer import TransferConfig
from botocore.exceptions import ClientError

STATE, MANIFEST = "migration_state.jsonl", "s3_inventory.csv"
if not os.path.exists(MANIFEST):
  raise SystemExit(f"{MANIFEST} not found -- run Step 3 first.")

cfg = TransferConfig(multipart_threshold=int(MULTIPART_THRESHOLD_MB * 1024 * 1024),
                     multipart_chunksize=16 * 1024 * 1024, max_concurrency=2)

done = set()
if os.path.exists(STATE):
  with open(STATE, encoding="utf-8") as f:
    for line in f:
      try:
        done.add(json.loads(line)["key"])
      except Exception:
        pass

with open(MANIFEST, encoding="utf-8") as f:
  plan = list(csv.DictReader(f))
pending = [r for r in plan if r["key"] not in done]

print(f"Planned {len(plan):,} | already done {len(done):,} | pending {len(pending):,} "
      f"({human(sum(int(r['size']) for r in pending))})")

lock = threading.Lock()
counts = {"copied": 0, "skipped": 0, "failed": 0, "bytes": 0}
failures = []


def migrate(row):
  key, dkey, size = row["key"], row["dest_key"], int(row["size"])

  try:                                   # already there with the same size?
    if dst.head_object(Bucket=TARGET_BUCKET, Key=dkey)["ContentLength"] == size:
      with lock:
        counts["skipped"] += 1
      return
  except ClientError:
    pass

  if DRY_RUN:
    with lock:
      counts["copied"] += 1
      counts["bytes"] += size
    return

  ctype = mimetypes.guess_type(key)[0] or "application/octet-stream"
  for attempt in range(1, RETRIES + 1):
    try:
      body = src.get_object(Bucket=SOURCE_BUCKET, Key=key)["Body"]
      try:
        dst.upload_fileobj(body, TARGET_BUCKET, dkey,
                           ExtraArgs={"ContentType": ctype}, Config=cfg)
      finally:
        body.close()
      with lock:
        counts["copied"] += 1
        counts["bytes"] += size
        with open(STATE, "a", encoding="utf-8") as sf:
          sf.write(json.dumps({"key": key, "size": size}) + "\n")
      return
    except Exception as e:
      if attempt == RETRIES:
        with lock:
          counts["failed"] += 1
          failures.append((key, repr(e)))
      else:
        time.sleep(2 ** attempt)


t0 = time.time()
print(f"\nStarting [{'DRY RUN' if DRY_RUN else 'LIVE'}] with {MAX_WORKERS} workers\n")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
  futures = [pool.submit(migrate, r) for r in pending]
  for i, fut in enumerate(as_completed(futures), 1):
    fut.result()
    if i % 100 == 0 or i == len(futures):
      el = time.time() - t0
      print(f"  {i:>7,}/{len(futures):,}  copied={counts['copied']:,} "
            f"skipped={counts['skipped']:,} failed={counts['failed']:,}  "
            f"{counts['bytes'] / el / 1024 / 1024 if el else 0:,.1f} MB/s")

print("\n" + "=" * 56)
print(f"Copied  {counts['copied']:,} ({human(counts['bytes'])})   "
      f"Skipped {counts['skipped']:,}   Failed {counts['failed']:,}")
print(f"Elapsed {(time.time() - t0) / 60:,.1f} min")

if failures:
  for key, err in failures[:5]:
    print(f"  {key[:50]}  {err[:80]}")
  print(f"\nRerun this cell to retry the {counts['failed']:,} failure(s).")
print("Migration completed")

## Step 6: Verify

Compares both sides by key and size. Etags are not compared: Supabase computes them differently
from S3 for multipart objects, so identical files can have different etags.

In [ ]:
#@title #Step 6: Verify the migration { display-mode: "form" }
def listing(client, bucket, prefix=""):
  out = {}
  for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for o in page.get("Contents", []):
      if not (o["Key"].endswith("/") and o["Size"] == 0):
        out[o["Key"]] = o["Size"]
  return out


source = listing(src, SOURCE_BUCKET, SOURCE_PREFIX)
target = listing(dst, TARGET_BUCKET, TARGET_PREFIX)
expected = {dest_key(k): v for k, v in source.items()}

missing = {k: v for k, v in expected.items() if k not in target}
mismatched = {k: (v, target[k]) for k, v in expected.items() if k in target and target[k] != v}

print("=" * 56)
print(f"Source       {len(source):,} files   {human(sum(source.values()))}")
print(f"Destination  {len(target):,} files   {human(sum(target.values()))}")
print(f"Missing      {len(missing):,}")
print(f"Wrong size   {len(mismatched):,}")

for k in list(missing)[:10]:
  print(f"  missing  {human(missing[k]):>10}  {k[:52]}")
for k, (a, b) in list(mismatched.items())[:10]:
  print(f"  size     {human(a):>10} != {human(b):>10}  {k[:40]}")

print("\nAll files present with matching sizes."
      if not missing and not mismatched else "\nRerun Step 5 to fill the gaps.")

## Alternative: rclone

Pick `rclone` in the first cell to use this instead. Same values as Step 1, nothing new to fill in.

In [ ]:
#@title #rclone: configure, dry-run, then copy { display-mode: "form" }
RUN_COPY = False #@param {type:"boolean"}
TRANSFERS = 4 #@param {type:"integer"}

import os

conf = f'''[aws]
type = s3
provider = AWS
access_key_id = {os.environ["AWS_ACCESS_KEY_ID"]}
secret_access_key = {os.environ["AWS_SECRET_ACCESS_KEY"]}
region = {os.environ["AWS_REGION"]}

[supabase]
type = s3
provider = Other
access_key_id = {os.environ["SUPABASE_ACCESS_KEY_ID"]}
secret_access_key = {os.environ["SUPABASE_SECRET_ACCESS_KEY"]}
endpoint = {os.environ["SUPABASE_S3_ENDPOINT"]}
region = {os.environ["SUPABASE_REGION"]}
'''

os.makedirs(os.path.expanduser("~/.config/rclone"), exist_ok=True)
with open(os.path.expanduser("~/.config/rclone/rclone.conf"), "w") as f:
  f.write(conf)
print(conf.replace(os.environ["AWS_SECRET_ACCESS_KEY"], "***")
          .replace(os.environ["SUPABASE_SECRET_ACCESS_KEY"], "***"))

SRC = f"aws:{os.environ['SOURCE_BUCKET']}/{os.environ.get('SOURCE_PREFIX', '')}".rstrip("/")
DST = f"supabase:{os.environ['TARGET_BUCKET']}/{os.environ.get('TARGET_PREFIX', '')}".rstrip("/")
print(f"{SRC}  ->  {DST}")

print("\n--- size of source (no data transferred) ---")
!rclone size "{SRC}"

print("\n--- dry run ---")
!rclone copy "{SRC}" "{DST}" --dry-run --transfers {TRANSFERS} --checkers 8

if RUN_COPY:
  !rclone copy "{SRC}" "{DST}" --progress --transfers {TRANSFERS} --checkers 8
  !rclone check "{SRC}" "{DST}" --size-only
  print("Migration completed")
else:
  print("\nRUN_COPY is off -- nothing transferred.")

## Troubleshooting

| Error | Fix |
|---|---|
| `SignatureDoesNotMatch` on the **source** | The AWS **secret** is wrong, not the permissions. Re-copy it or create a new key pair. Check it is 40 chars with no quotes or stray spaces |
| `403 Forbidden` on the **source** | The AWS key is valid but not authorized. Attach the IAM policy in Step 2, check `SOURCE_BUCKET` spelling, and confirm `AWS_REGION` matches the bucket |
| `SignatureDoesNotMatch` | Regenerate the key pair; check the region matches the project and the endpoint ends in `/storage/v1/s3` |
| `NoSuchBucket` on a bucket you can see | Client needs path addressing (already set in Step 2) |
| `XAmzContentSHA256Mismatch` / `501` | botocore checksum headers (already disabled in Step 2) |
| `413 Payload too large` | File over the bucket limit. Raise it in Storage &rarr; Buckets |
| Stalls or `429` | Lower `MAX_WORKERS` to 4 |

[S3 compatibility](https://supabase.com/docs/guides/storage/s3/compatibility) &middot;
[S3 auth](https://supabase.com/docs/guides/storage/s3/authentication) &middot;
[Copying objects](https://supabase.com/docs/guides/self-hosting/copy-from-platform-s3) &middot;
[File limits](https://supabase.com/docs/guides/storage/uploads/file-limits)